In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers
import nltk
import nltk

In [ ]:
print(tf.__version__)

2.19.0


In [ ]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau

In [ ]:
from nltk.corpus import gutenberg

In [ ]:
nltk.download('gutenberg')

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


True

# load data

In [ ]:
# Load book text
text = gutenberg.raw('austen-emma.txt')

# Lowercase
text = text.lower()

print("total characters:", len(text))

total characters: 887071


In [ ]:
text[:11]

'[emma by ja'

In [ ]:
vocab_size = 10000  #vocaboly size
seq_length = 30


In [ ]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts([text])



In [ ]:
total_words = min(vocab_size, len(tokenizer.word_index) + 1)

print("Vocabulary size:", total_words)


Vocabulary size: 7234


In [ ]:
len(text)

887071

In [ ]:
input_sequences = []

token_list = tokenizer.texts_to_sequences([text])[0]

for i in range(seq_length, len(token_list)):
    n_gram_sequence = token_list[i-seq_length:i+1]
    input_sequences.append(n_gram_sequence)

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]



In [ ]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (161071, 30)
Shape of y: (161071,)


In [ ]:
embedding_dim = 256
lstm_units = 512

model = tf.keras.Sequential([

    layers.Embedding(input_dim=total_words,
                     output_dim=embedding_dim,
                     input_length=seq_length),

    layers.LSTM(lstm_units, return_sequences=True),
    layers.LayerNormalization(),
    layers.Dropout(0.3),

    # layers.LSTM(lstm_units,return_sequences=True),
    # layers.Dropout(0.3),


    layers.LSTM(lstm_units),
    layers.Dropout(0.3),

    layers.Dense(512, activation='relu'),

    layers.Dense(total_words, activation='softmax')
])



In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_1           │ ?                      │   0 (unbuilt) │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',      # watch validation loss
    patience=3,              # wait 3 epochs before stopping
    restore_best_weights=True,
    verbose=1
)

In [ ]:
history = model.fit(
    X,
    y,
    batch_size=64,
    epochs=20,
    validation_split=0.1,
    callbacks=[early_stop,lr_scheduler]
)

Epoch 1/20
2266/2266 ━━━━━━━━━━━━━━━━━━━━ 71s 30ms/step - accuracy: 0.0391 - loss: 6.3820 - val_accuracy: 0.0685 - val_loss: 6.0053 - learning_rate: 0.0010
Epoch 2/20
2266/2266 ━━━━━━━━━━━━━━━━━━━━ 67s 29ms/step - accuracy: 0.0770 - loss: 5.8059 - val_accuracy: 0.0993 - val_loss: 5.6807 - learning_rate: 0.0010
Epoch 3/20
2266/2266 ━━━━━━━━━━━━━━━━━━━━ 67s 29ms/step - accuracy: 0.1043 - loss: 5.4787 - val_accuracy: 0.1065 - val_loss: 5.6217 - learning_rate: 0.0010
Epoch 4/20
2266/2266 ━━━━━━━━━━━━━━━━━━━━ 67s 30ms/step - accuracy: 0.1175 - loss: 5.2880 - val_accuracy: 0.1155 - val_loss: 5.5800 - learning_rate: 0.0010
Epoch 5/20
2266/2266 ━━━━━━━━━━━━━━━━━━━━ 67s 30ms/step - accuracy: 0.1270 - loss: 5.1515 - val_accuracy: 0.1173 - val_loss: 5.5578 - learning_rate: 0.0010
Epoch 6/20
2266/2266 ━━━━━━━━━━━━━━━━━━━━ 67s 29ms/step - accuracy: 0.1335 - loss: 5.0369 - val_accuracy: 0.1225 - val_loss: 5.5916 - learning_rate: 0.0010
Epoch 7/20
2265/2266 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accurac

In [ ]:
model.save("word_suggestion_model.keras")

import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [ ]:
def generate_text(seed_text, next_words=20, temperature=0.8):
    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=seq_length, padding='pre')

        predictions = model.predict(token_list, verbose=0)

        predictions = np.log(predictions) / temperature
        exp_preds = np.exp(predictions)
        predictions = exp_preds / np.sum(exp_preds)

        predicted_index = np.random.choice(range(total_words), p=predictions[0])

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

In [ ]:
print(generate_text("i am going to", next_words=15, temperature=0.8))

i am going to have going to both any subject and like him you know he was agreed to


In [ ]:
from google.colab import files

files.download("word_suggestion_model.keras")
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(generate_text("i am going to", next_words=15, temperature=5.5))

i am going to appeared how minority style enter dirty exertion southward saturday darted would encounter exactly profusion seven
